<a href="https://colab.research.google.com/github/Omotayo-Emmanuel/AyomideDataAnalysisPortfolio/blob/main/DSN_HACKATHON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount the drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### Mount Google Drive

This cell mounts my Google Drive to the Colab environment, allowing the notebook to access files stored in my Drive.

In [ ]:
import os, json, time, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from autogluon.tabular import TabularPredictor

#### Import Libraries

Essential libraries like `os`, `json`, `time`, `hashlib`, `pathlib`, `numpy`, `pandas`, `matplotlib.pyplot`, and `autogluon.tabular.TabularPredictor` are imported. These are fundamental for data manipulation, numerical operations, plotting, and machine learning with AutoGluon.

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "autogluon.tabular[tabicl,nori,tabdpt]==1.6.3",
])
print("Installed. NOW: Runtime menu → Restart session")

Installed. NOW: Runtime menu → Restart session


#### Install AutoGluon

This cell ensures that the AutoGluon library, specifically `autogluon.tabular`, is installed and updated to version `1.6.3`. It uses `subprocess.check_call` to run `pip install` silently. Then restart of the Colab runtime which is recommeded after the installation is required

In [ ]:
ROOT = Path("/content/drive/MyDrive/DSN_collab_bootcamp")
RAW  = ROOT / "data" / "raw"
OUT = ROOT / "artifcats" / f"run_{int(time.time())}"
OUT.mkdir(parents = True, exist_ok = True)
SUB_DIR = ROOT / "submissions"
SUB_DIR.mkdir(parents = True, exist_ok = True)

print("Ready. Output dir:", OUT)

Ready. Output dir: /content/drive/MyDrive/DSN_collab_bootcamp/artifcats/run_1790511542


#### Define File Paths

This is where the file paths for managing data and outputs are defined
- `ROOT`: The base directory for the project on Google Drive.
- `RAW`: Directory for raw data (though not explicitly used for loading in later cells, it sets a standard).
- `OUT`: An output directory named with a timestamp to store model artifacts for each run.
- `SUB_DIR`: Directory for saving submission files.

`OUT.mkdir` and `SUB_DIR.mkdir` ensure these directories exist, creating them if necessary.

In [ ]:
ID_COL, TARGET = "id", "total_sales"

train = pd.read_csv(ROOT /"train.csv")
test = pd.read_csv(ROOT / "test.csv")
sample = pd.read_csv(ROOT / "sample_submission.csv")

#### Load Datasets

The `train.csv`, `test.csv`, and `sample_submission.csv` files are loaded into pandas DataFrames. These files contain the training data, test data for prediction, and the format required for the submission file, respectively. `ID_COL` and `TARGET` are also defined for clarity.

The Assert keyword is used to test assumptions in the code. Testing data integrity, ensuring shapes, column names or values  meet expectations before further processing

In [ ]:
assert train.shape == (6818, 13)
assert test.shape == (1705, 12)
assert sample.columns.tolist() == [ID_COL, TARGET]
assert sample[ID_COL].equals(test[ID_COL])

#### Data Integrity Checks

`assert` statements are used to verify the shapes of the loaded DataFrames and the column names of the sample submission.

In [ ]:
y = train[TARGET].to_numpy(float)
print("Data Loaded. Train:", train.shape, "| Test:", test.shape)
print("Test product seen rate:",
      round(test['product_code'].isin(train['product_code']).mean(), 5))

Data Loaded. Train: (6818, 13) | Test: (1705, 12)
Test product seen rate: 0.99648


### 2. Feature Engineering

This section defines and applies a feature engineering function to both the training and testing datasets, creating new features and transforming existing ones to improve model performance.

#### Initial Data Information

The target variable `y` is extracted from the training data. The shapes of the training and testing datasets are printed, along with the "test product seen rate," which indicates the proportion of `product_code` values in the test set that are also present in the training set. This gives an idea of how much novelty the model will encounter.

In [ ]:
def make_features(frame):
  out = frame.drop(columns = [ID_COL, TARGET], errors ="ignore").copy()

  # Using the Missingness as information - Not as Zero
  out["product_weight_missing"] = out["product_weight_kg"].isnull().astype("int8")
  out["store_size_missing"] = out["store_size"].isnull().astype("int8")
  out["is_zero_visibility"] = out["shelf_visibility"].eq(0).astype("int8")


  # Text normalization
  out["product_category"] = (out["product_category"].astype("string")
                              .str.strip().str.lower().str.title())
  out["fat_content"] = (out["fat_content"].astype("string").str.strip()
                        .replace({"LF": "Low Fat", "low fat": "Low Fat, reg: Regular"}))

  for col in ["product_code", "store_code", "store_size",
              "store_location_tier", "store_format"]:
              out[col] = out[col].astype("string").str.strip().fillna("Unknown")


  # Log transforms on the features and also jeep the raw versions too
  out["log_product_price"] = np.log1p(out["product_price"])
  out["log_shelf_visibility"] = np.log1p(out["shelf_visibility"])
  return out

X = make_features(train)
X_test = make_features(test)




#### `make_features` Function

This function takes a DataFrame and performs several feature engineering steps:
- **Missingness as Information**: Creates binary features (`_missing`) to indicate if `product_weight_kg` or `store_size` were originally null, treating missingness as a potentially informative signal.
- **Zero Visibility**: Creates a binary feature `is_zero_visibility` for `shelf_visibility` being zero.
- **Text Normalization**: Standardizes categorical features like `product_category` (strip, lowercase, title case) and `fat_content` (replaces abbreviations like 'LF' and 'low fat' with 'Low Fat').
- **Type Conversion and Imputation**: Converts several categorical columns to `string` type, strips whitespace, and imputes missing values with 'Unknown'.
- **Log Transforms**: Applies `np.log1p` (log(1+x)) to `product_price` and `shelf_visibility` to handle skewed distributions, creating `log_product_price` and `log_shelf_visibility`.

The function returns the transformed DataFrame. It's applied to both `train` to create `X` and `test` to create `X_test`.

In [ ]:
assert X.shape == (6818, 16)
assert X_test.shape == (1705, 16)
assert X.columns.tolist() == X_test.columns.tolist()
print("Features:", X.shape[1])
print(X.columns.tolist())

Features: 16
['product_code', 'product_weight_kg', 'fat_content', 'shelf_visibility', 'product_category', 'product_price', 'store_code', 'store_age_years', 'store_size', 'store_location_tier', 'store_format', 'product_weight_missing', 'store_size_missing', 'is_zero_visibility', 'log_product_price', 'log_shelf_visibility']


### 3. Custom Validation Strategy (Matched Product Coverage)

This section implements a sophisticated custom validation splitting strategy designed to ensure that the validation set's product coverage closely matches that of the test set, which is crucial for robust evaluation in this specific problem context.

#### Feature Integrity Checks

Assertions are used again to confirm that the `X` and `X_test` DataFrames have the expected shapes and identical column names after feature engineering. The number of features and their names are also printed, verifying the feature creation process.

MATCHED product coverage validation



#### Product and Target Stratification

These cells prepare data for the custom validation split:
- `products`: Extracts `product_code` from the training data.
- `counts`: Calculates the frequency of each product code.
- `row_product_count`: Maps each training row to the count of its corresponding product code.
- `target_strata`: Divides the target variable `y` into 10 quantiles, creating strata for stratified sampling.
- `strata_counts`, `strata_values`: Summarize the distribution of these target strata.

In [ ]:
products = train["product_code"].fillna("MISSING").astype(str).to_numpy()
counts = pd.Series(products).value_counts()
row_product_count = pd.Series(products).map(counts).to_numpy(int)



In [ ]:
target_strata = np.asarray(pd.qcut(y, q=10, labels=False, duplicates="drop"), dtype=int)
strata_counts = pd.Series(target_strata).value_counts().sort_index()
strata_values = strata_counts.index.tolist()


In [ ]:
n_rows = len(train)
n_valid = int(round(0.20 * n_rows))
test_seen_rate = test["product_code"].isin(train["product_code"]).mean()
target_unseen_rows = int(round((1 - test_seen_rate) * n_valid))
singleton_idx = np.flatnonzero(row_product_count == 1)

#### Define Validation Parameters

Key parameters for the validation split are set:
- `n_rows`: Total number of rows in the training data.
- `n_valid`: Number of rows to allocate for the validation set (20% of `n_rows`).
- `test_seen_rate`: Reconfirms the rate of test products seen in the training data.
- `target_unseen_rows`: Estimates how many validation rows should ideally represent products not seen in the remaining training split (to mimic the test set).
- `singleton_idx`: Identifies indices of products that appear only once in the training data, which are important for handling unseen products.

In [ ]:
def make_quotas(total):
    raw = strata_counts / strata_counts.sum() * total
    q = np.floor(raw).astype(int)
    rem = int(total - q.sum())
    if rem:
        for s in (raw - q).sort_values(ascending=False, kind="stable").index[:rem]:
            q.loc[s] += 1
    return q.to_dict()

#### `make_quotas` Function

This helper function calculates the desired count of items per stratum given a total number of items. It ensures that the proportions of each stratum are maintained as closely as possible when sampling a subset of the data.

In [ ]:
def choose_matched_validation(seed):
    rng = np.random.default_rng(seed)
    q_all = make_quotas(n_valid)
    q_unseen = make_quotas(target_unseen_rows)
    selected, sel_set = [], set()
    sel_per_stratum = {s: 0 for s in strata_values}

    # Stage 1: reserve singletons for unseen products
    for s in strata_values:
        cand = singleton_idx[target_strata[singleton_idx] == s]
        take = min(int(q_unseen[s]), len(cand))
        if take:
            ch = rng.choice(cand, size=take, replace=False)
            selected.extend(ch.tolist()); sel_set.update(ch.tolist())
            sel_per_stratum[s] += take

    need = target_unseen_rows - len(selected)
    if need:
        rem = np.array([i for i in singleton_idx if i not in sel_set])
        extra = rng.choice(rem, size=need, replace=False)
        selected.extend(extra.tolist()); sel_set.update(extra.tolist())
        for i in extra:
            sel_per_stratum[target_strata[i]] += 1

    # Stage 2: repeated products — keep at least 1 row in train
    capacity = {p: int(c - 1) for p, c in counts.items() if c >= 2}

    def take(cand, required):
        out = []
        for i in rng.permutation(cand):
            if i in sel_set: continue
            p = products[i]
            if capacity.get(p, 0) <= 0: continue
            out.append(int(i)); sel_set.add(int(i))
            capacity[p] -= 1
            if len(out) == required: break
        return out

    for s in strata_values:
        needed = max(0, int(q_all[s] - sel_per_stratum[s]))
        if needed:
            cand = np.flatnonzero((target_strata == s) & (row_product_count >= 2))
            ch = take(cand, needed)
            selected.extend(ch); sel_per_stratum[s] += len(ch)

    rem = n_valid - len(selected)
    if rem:
        selected.extend(take(np.flatnonzero(row_product_count >= 2), rem))

    valid_idx = np.sort(np.array(selected, dtype=int))
    fit_idx = np.setdiff1d(np.arange(n_rows), valid_idx)
    known_rate = float(pd.Series(products[valid_idx]).isin(set(products[fit_idx])).mean())

    assert len(fit_idx) == 5454 and len(valid_idx) == 1364
    assert abs(known_rate - test_seen_rate) < 0.001
    return fit_idx, valid_idx, known_rate


#### `choose_matched_validation` Function

This is the core function for creating the custom train-validation split. Its goal is to create a validation set that statistically mirrors the `test_seen_rate` of the actual test set, especially concerning `product_code`.

- **Stage 1 (Unseen Products)**: Prioritizes selecting singleton products (those appearing once) for the validation set, ensuring that a representative number of 'unseen' products (relative to the training split) are included, stratified by target quantiles.
- **Stage 2 (Repeated Products)**: Fills the remaining validation quota by selecting from products that appear multiple times, ensuring that each selected product still has at least one instance left in the training set.

Assertions at the end confirm the sizes of the `fit_idx` (training indices) and `valid_idx` (validation indices) and that the `known_rate` of products in the validation set matches the `test_seen_rate`.

In [ ]:
OUTER_SEEDS = [2026, 123, 31415]
OUTER_SPLITS = {}
for s in OUTER_SEEDS:
    fit_idx, valid_idx, rate = choose_matched_validation(s)
    OUTER_SPLITS[s] = {"fit_idx": fit_idx, "valid_idx": valid_idx}
    print(f"seed {s}: valid_rate={rate:.6f} (test={test_seen_rate:.6f})")

seed 2026: valid_rate=0.996334 (test=0.996481)
seed 123: valid_rate=0.996334 (test=0.996481)
seed 31415: valid_rate=0.996334 (test=0.996481)


### 4. AutoGluon Outer Validation (Lever 1)

This section performs model training and evaluation using AutoGluon on the previously defined outer validation splits. This helps assess the model's generalization performance.

#### Apply Validation Splits for Multiple Seeds

The `choose_matched_validation` function is called with multiple `OUTER_SEEDS` (2026, 123, 31415). This creates different train-validation splits based on varying random seeds, allowing for a more robust evaluation of the model's performance by reducing the impact of a single random split. The `fit_idx` and `valid_idx` for each seed are stored in `OUTER_SPLITS`.

In [ ]:
PRESET = "extreme"
TIME_PER_OUTER = 1800      # 10 min per fold
TIME_FULL = 1800           # 10 min for final refit

#### Define AutoGluon Configuration

Configuration variables for AutoGluon are set:
- `PRESET`: Specifies the AutoGluon preset to use (e.g., "extreme" for high quality).
- `TIME_PER_OUTER`: Time limit in seconds for training each outer validation fold.
- `TIME_FULL`: Time limit in seconds for training the final model on the full dataset.

In [ ]:
def rmse(a, p):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(p)) ** 2)))


#### RMSE Evaluation Metric

This small function `rmse` calculates the Root Mean Squared Error, which is the chosen evaluation metric for this regression problem. It measures the average magnitude of the errors.

In [ ]:
# --- Step 1: Outer validation on 3 seeds ---
outer_results = []
for seed in OUTER_SEEDS:
    fit_idx = OUTER_SPLITS[seed]["fit_idx"]
    valid_idx = OUTER_SPLITS[seed]["valid_idx"]

    fold_train = make_features(train.iloc[fit_idx].reset_index(drop=True))
    fold_train[TARGET] = y[fit_idx]
    fold_valid = make_features(train.iloc[valid_idx].reset_index(drop=True))

    model_dir = OUT / f"autogluon_seed_{seed}"
    predictor = TabularPredictor(
        label=TARGET, problem_type="regression",
        eval_metric="root_mean_squared_error",
        path=str(model_dir), verbosity=2,
    )
    t0 = time.time()
    predictor.fit(
        train_data=fold_train, presets=PRESET,
        time_limit=TIME_PER_OUTER, num_gpus=1,
        num_cpus=max(1, os.cpu_count() or 1),
        dynamic_stacking=False,
        random_seed=seed,
    )
    preds = np.maximum(0.0, predictor.predict(fold_valid).to_numpy(float))
    score = rmse(y[valid_idx], preds)
    outer_results.append({"seed": seed, "rmse": score,
                          "best_model": predictor.model_best,
                          "fit_time": time.time() - t0})
    print(f"seed {seed}: RMSE={score:.2f} | best={predictor.model_best}")



Preset alias specified: 'extreme' maps to 'extreme_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.06 GB / 12.67 GB (79.4%)
Disk Space Avail:   61.19 GB / 112.64 GB (54.3%)
Presets specified: ['extreme']
Using hyperparameters preset: hyperparameters='commercial_2026_08_05'
Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "/content/drive/MyDrive/DSN_collab_bootcamp/artifcats/run_1790468695/autogluon_seed_2026"
Train Data Rows:    5454
Train Data Columns: 16
Label Column:       total_sales
Problem Type:       regres

seed 2026: RMSE=1062.21 | best=WeightedEnsemble_L2


Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "/content/drive/MyDrive/DSN_collab_bootcamp/artifcats/run_1790468695/autogluon_seed_123"
Train Data Rows:    5454
Train Data Columns: 16
Label Column:       total_sales
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    10556.34 MB
	Train Data (Original)  Memory Usage: 2.40 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 4 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGener

seed 123: RMSE=1065.75 | best=WeightedEnsemble_L2


Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "/content/drive/MyDrive/DSN_collab_bootcamp/artifcats/run_1790468695/autogluon_seed_31415"
Train Data Rows:    5454
Train Data Columns: 16
Label Column:       total_sales
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    10559.80 MB
	Train Data (Original)  Memory Usage: 2.40 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 4 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGen

seed 31415: RMSE=1091.55 | best=WeightedEnsemble_L2


#### Outer Loop Training and Evaluation

This loop iterates through each of the defined `OUTER_SEEDS`:
1.  **Split Data**: Uses the `fit_idx` and `valid_idx` to create training and validation sets for the current fold.
2.  **Initialize Predictor**: An `TabularPredictor` instance from AutoGluon is initialized for each fold, configured for regression with RMSE as the evaluation metric. A unique path is created for each model to save its artifacts.
3.  **Train Model**: The `predictor.fit` method trains the model using the `fold_train` data, the specified `PRESET`, `time_limit`, and GPU/CPU configuration. `dynamic_stacking` is set to `False` and `random_seed` is passed for reproducibility of each fold.
4.  **Predict and Evaluate**: Predictions are made on the `fold_valid` set, ensuring predictions are non-negative (`np.maximum(0.0, ...)`). The `rmse` function is used to calculate the score for the current fold.
5.  **Store Results**: The seed, RMSE score, best model found by AutoGluon, and fitting time are stored in `outer_results`.

In [ ]:
outer_df = pd.DataFrame(outer_results)
outer_df.to_csv(OUT / "outer_results.csv", index=False)
print("\nMean outer RMSE:", round(outer_df["rmse"].mean(), 2))
print("Worst outer RMSE:", round(outer_df["rmse"].max(), 2))


Mean outer RMSE: 1073.17
Worst outer RMSE: 1091.55


### 5. AutoGluon Full Model Training and Initial Submission

After evaluating the model's performance on various validation splits, this section trains a final model on the entire training dataset and generates the initial submission file.

#### Summarize Outer Validation Results

The `outer_results` list is converted into a pandas DataFrame and saved to a CSV file. The mean and worst RMSE scores across all outer validation folds are printed. This provides an overall indication of the model's performance variability and generalizability.

In [ ]:
# --- Step 2: Full-data refit and save the artifact ---
full_train = make_features(train)
full_train[TARGET] = y
full_test = make_features(test)

#### Prepare Full Data

The `make_features` function is applied to the entire `train` DataFrame to create `full_train` and to the `test` DataFrame to create `full_test`. The `TARGET` variable is added back to `full_train`.

In [ ]:
full_dir = OUT / "autogluon_full"
predictor_full = TabularPredictor(
    label=TARGET, problem_type="regression",
    eval_metric="root_mean_squared_error",
    path=str(full_dir), verbosity=2,
)

#### Initialize Full Predictor

A new `TabularPredictor` instance is initialized for training on the full dataset. It uses the same configuration (label, problem type, metric) as the outer validation predictors, but saves its artifacts to a new directory named `autogluon_full`.

In [ ]:
predictor_full.fit(
    train_data=full_train, presets=PRESET,
    time_limit=TIME_FULL, num_gpus=1,
    num_cpus=max(1, os.cpu_count() or 1),
    dynamic_stacking=False,
)
print("Full model saved at:", full_dir)


Preset alias specified: 'extreme' maps to 'extreme_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.30 GB / 12.67 GB (81.3%)
Disk Space Avail:   60.98 GB / 112.64 GB (54.1%)
Presets specified: ['extreme']
Using hyperparameters preset: hyperparameters='commercial_2026_08_05'
Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "/content/drive/MyDrive/DSN_collab_bootcamp/artifcats/run_1790468695/autogluon_full"
Train Data Rows:    6818
Train Data Columns: 16
Label Column:       total_sales
Problem Type:       regression


Full model saved at: /content/drive/MyDrive/DSN_collab_bootcamp/artifcats/run_1790468695/autogluon_full


#### Fit Full Predictor

The `predictor_full.fit` method trains the model on the `full_train` dataset with the `extreme` preset and the `TIME_FULL` limit. This model is intended to be the final model used for making predictions on the unseen test data. The path where the full model is saved is also printed.

In [ ]:
# --- Step 3: Predict and export ---
preds_full = np.maximum(0.0, predictor_full.predict(full_test).to_numpy(float))


#### Predict on Test Data

Once the full model is trained, it generates predictions (`preds_full`) on the `full_test` dataset. `np.maximum(0.0, ...)` is used again to ensure all predictions are non-negative.

In [ ]:
submission = sample.copy()
submission[TARGET] = preds_full
assert submission.shape == (1705, 2)
assert submission[ID_COL].equals(test[ID_COL])
assert np.isfinite(submission[TARGET]).all()

#### Create and Verify Submission

The `sample_submission` DataFrame is copied, and the generated `preds_full` are assigned to the `TARGET` column. Assertions confirm the shape of the submission file, that the `id` column matches the test set, and that all predictions are finite numbers.

In [ ]:
sub_path = SUB_DIR / f"submission_{OUT.name}.csv"
submission.to_csv(sub_path, index=False)

#### Save Submission File

The final submission DataFrame is saved to a CSV file in the `SUB_DIR` with a filename

In [ ]:
print("\nSUBMISSION READY")
print("Path:", sub_path)
print("Prediction mean:", round(preds_full.mean(), 2))
print("Prediction std :", round(preds_full.std(), 2))
print("Prediction p95 :", round(np.quantile(preds_full, 0.95), 2))
print(submission.head())


SUBMISSION READY
Path: /content/drive/MyDrive/DSN_collab_bootcamp/submissions/submission_run_1790468695.csv
Prediction mean: 2196.05
Prediction std : 1276.39
Prediction p95 : 4247.49
          id  total_sales
0  row_00009  2873.099365
1  row_00015  4366.952637
2  row_00019  2788.874756
3  row_00020  3174.314209
4  row_00023  2478.790771


### 6. Ensemble with Additional Refits (Lever 2)

This section explores an ensembling strategy by performing additional model fits and then averaging their predictions with the initial run's predictions. This can often lead to more robust and accurate results.

#### Print Submission Details

This cell provides a summary of the generated submission file, including its path, the mean, standard deviation, and 95th percentile of the predictions, and the first few rows of the submission DataFrame. This offers a quick check on the distribution of the final predictions.

In [ ]:
# Fixed list of seeds to run. Seed 0 = your existing Lever 1 run.
SEEDS_TO_RUN = [7, 2026]   # two additional runs

# Store results as we go
all_runs = []   # will hold {"seed": int, "outer_mean": float, "test_preds": np.array}


#### Setup for Additional Runs

`SEEDS_TO_RUN` is defined (though not directly used in the following loop, it signifies an intent for multiple runs). An empty list `all_runs` is initialized to store results from various runs.

In [ ]:
SUB_DIR = Path("/content/drive/MyDrive/DSN_collab_bootcamp/submissions")
OUT_BASE = Path("/content/drive/MyDrive/DSN_collab_bootcamp/artifcats")

#### Define Paths for Lever 2

`SUB_DIR` and `OUT_BASE` are redefined here. `OUT_BASE` specifically points to the base directory for AutoGluon artifacts.

In [ ]:
# ---- existing run (Lever 1) ----
run1_path = SUB_DIR / "submission_run_1790468695.csv"
assert run1_path.exists(), f"Missing: {run1_path}"
run1_preds = pd.read_csv(run1_path)["total_sales"].to_numpy()
print(f"Run 1 loaded: mean={run1_preds.mean():.2f}, std={run1_preds.std():.2f}")

Run 1 loaded: mean=2196.05, std=1276.39


#### Load Initial Run Predictions

The predictions from the first full model run (referred to as "Lever 1") are loaded from its saved submission CSV. This allows it to be combined with subsequent runs for ensembling.

In [ ]:
# ---- Two more plain refits (no random_seed) ----
new_preds_list = []

for i in range(2):
    print(f"\n{'='*60}\nRefit #{i+2}\n{'='*60}")
    seed_dir = OUT_BASE / f"lever2_refit{i}_{int(time.time())}"

    p = TabularPredictor(
        label=TARGET, problem_type="regression",
        eval_metric="root_mean_squared_error",
        path=str(seed_dir), verbosity=2,
    )
    p.fit(
        train_data=full_train, presets="extreme",
        time_limit=1800, num_gpus=1,
        num_cpus=max(1, os.cpu_count() or 1),
        dynamic_stacking=False,
    )

    preds = np.maximum(0.0, p.predict(full_test).to_numpy(float))
    new_preds_list.append(preds)
    print(f"  refit #{i+2}: mean={preds.mean():.2f}, std={preds.std():.2f}")

    # Save this refit individually
    refit_sub = sample.copy()
    refit_sub[TARGET] = preds
    refit_sub.to_csv(SUB_DIR / f"submission_refit{i}.csv", index=False)


Preset alias specified: 'extreme' maps to 'extreme_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.79 GB / 12.67 GB (85.2%)
Disk Space Avail:   61.55 GB / 112.64 GB (54.6%)
Presets specified: ['extreme']
Using hyperparameters preset: hyperparameters='commercial_2026_08_05'



Refit #2


Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "/content/drive/MyDrive/DSN_collab_bootcamp/artifcats/lever2_refit0_1790512213"
Train Data Rows:    6818
Train Data Columns: 16
Label Column:       total_sales
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    11029.35 MB
	Train Data (Original)  Memory Usage: 3.00 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 4 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	

  refit #2: mean=2195.72, std=1275.86

Refit #3


Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "/content/drive/MyDrive/DSN_collab_bootcamp/artifcats/lever2_refit1_1790512400"
Train Data Rows:    6818
Train Data Columns: 16
Label Column:       total_sales
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    10709.15 MB
	Train Data (Original)  Memory Usage: 3.00 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 4 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	

  refit #3: mean=2195.77, std=1276.32


#### Perform Additional Refits

This loop performs two more AutoGluon model fits (`Refit #2` and `Refit #3`) on the `full_train` data, similar to the initial full model training. Importantly, these runs do *not* specify a `random_seed` in the `predictor.fit` call, which means AutoGluon's internal randomness will differ, potentially leading to slightly different models and predictions. Each set of predictions is saved individually and added to `new_preds_list`.

In [ ]:
# ---- Check if the runs actually differ ----
print(f"\n{'='*60}\nRun comparison\n{'='*60}")
for i, preds in enumerate(new_preds_list):
    diff = np.abs(preds - run1_preds).mean()
    print(f"  refit #{i+2} vs run1: mean abs diff = {diff:.2f}")


Run comparison
  refit #2 vs run1: mean abs diff = 3.34
  refit #3 vs run1: mean abs diff = 1.98


#### Compare Runs

This cell calculates the mean absolute difference between the predictions from `Refit #2` and `Refit #3` and the initial `run1_preds`. This helps to quantify how much the predictions from different AutoGluon runs vary, confirming that they are indeed distinct enough to be useful for ensembling.

In [ ]:
# ---- Average the 3 runs ----
stacked = np.stack([run1_preds] + new_preds_list, axis=0)
averaged = stacked.mean(axis=0)
print(f"\nAveraged prediction: mean={averaged.mean():.2f}, std={averaged.std():.2f}")




Averaged prediction: mean=2195.85, std=1276.18


### 7. AutoGluon "Foundation Ensemble" and Final Submission

This section demonstrates another approach to leveraging AutoGluon's ensembling capabilities, explicitly allowing it to build a more complex ensemble of models, and then generates a final submission.

#### Average Predictions

The predictions from `run1_preds` and the `new_preds_list` (from `Refit #2` and `Refit #3`) are stacked together and then averaged element-wise. This creates a simple ensemble prediction by averaging the outputs of multiple models, which often yields better generalization than a single model.

In [ ]:
seed_dir = OUT_BASE / f"foundation_{int(time.time())}"
p = TabularPredictor(
    label=TARGET, problem_type="regression",
    eval_metric="root_mean_squared_error",
    path=str(seed_dir), verbosity=2,
)
p.fit(
    train_data=full_train, presets="extreme",
    time_limit=1800, num_gpus=1,
    num_cpus=max(1, os.cpu_count() or 1),
    dynamic_stacking=False,
)

# Check the ensemble actually includes multiple models this time
lb = p.leaderboard(silent=True)
print("\nTop 5 models:")
print(lb.head()[["model", "score_val"]])

preds = np.maximum(0.0, p.predict(full_test).to_numpy(float))
sub = sample.copy(); sub[TARGET] = preds
path = SUB_DIR / "submission_foundation_ensemble.csv"
sub.to_csv(path, index=False)
print(f"\nSaved: {path}")
print(f"Pred mean: {preds.mean():.2f}, std: {preds.std():.2f}")


Preset alias specified: 'extreme' maps to 'extreme_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.52 GB / 12.67 GB (83.0%)
Disk Space Avail:   61.20 GB / 112.64 GB (54.3%)
Presets specified: ['extreme']
Using hyperparameters preset: hyperparameters='commercial_2026_08_05'
Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "/content/drive/MyDrive/DSN_collab_bootcamp/artifcats/foundation_1790514277"
Train Data Rows:    6818
Train Data Columns: 16
Label Column:       total_sales
Problem Type:       regression
Preproce

nori.pt: reconstructing file:   0%|          |  0.00B /  117MB            

nori.pt: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/534 [00:00<?, ?B/s]

	Fitting 1 model on all data | Fitting with cpus=2, gpus=1, mem=2.8/9.6 GB
	-1073.2205	 = Validation score   (-root_mean_squared_error)
	290.09s	 = Training   runtime
	289.45s	 = Validation runtime
Fitting model: TabICLv2_BAG_L1 ... Training model for up to 1509.17s of the 1509.15s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=1, gpus=1)


...

	Fitting 1 model on all data | Fitting with cpus=2, gpus=1, mem=3.1/9.6 GB
	-1073.9253	 = Validation score   (-root_mean_squared_error)
	24.38s	 = Training   runtime
	15.9s	 = Validation runtime
Fitting model: TabDPT-Turbo_BAG_L1 ... Training model for up to 1484.08s of the 1484.07s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=1, gpus=1)


tabdpt1_2.safetensors: reconstructing file:   0%|          |  0.00B /  254MB            

tabdpt1_2.safetensors: downloading bytes:           |  0.00B            

	Fitting 1 model on all data | Fitting with cpus=2, gpus=1, mem=2.6/9.6 GB
	-1073.6519	 = Validation score   (-root_mean_squared_error)
	93.7s	 = Training   runtime
	82.55s	 = Validation runtime
Fitting model: PrepLightGBM_BAG_L1 ... Training model for up to 1388.63s of the 1388.61s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=1, gpus=0)
	-1074.4023	 = Validation score   (-root_mean_squared_error)
	163.41s	 = Training   runtime
	2.29s	 = Validation runtime
Fitting model: LightGBM_r8_BAG_L1 ... Training model for up to 1221.36s of the 1221.35s of remaining time.
	Skipping LightGBM_r8_BAG_L1 because a fit constraint is not satisfied: ag.min_rows=50000, but the data has only 6818 rows.
Fitting model: CatBoost_BAG_L1 ... Training model for up to 1221.35s of the 1221.33s of remaining time.
	Skipping CatBoost_BAG_L1 because a fit constraint is not satisfied: ag.min_rows=50000, but the data has only 6818 rows.
Fit


Top 5 models:
                 model    score_val
0  WeightedEnsemble_L2 -1071.225695
1      Nori-30M_BAG_L1 -1073.220497
2  TabDPT-Turbo_BAG_L1 -1073.651867
3      TabICLv2_BAG_L1 -1073.925350
4  PrepLightGBM_BAG_L1 -1074.402288

Saved: /content/drive/MyDrive/DSN_collab_bootcamp/submissions/submission_foundation_ensemble.csv
Pred mean: 2203.75, std: 1314.63


#### Train Foundation Ensemble and Save Submission

- **Initialize Predictor**: A new `TabularPredictor` is initialized, again using the `extreme` preset and the full training data.
- **Fit Model**: The `predictor.fit` method trains the model. Because AutoGluon's default behavior with `presets='extreme'` (or `extreme_quality`) is to build an ensemble of diverse models, this run aims to produce a more sophisticated ensemble internally.
- **Leaderboard**: `p.leaderboard(silent=True)` is called to inspect the top-performing individual models that AutoGluon considered during its internal ensembling process. This verifies that AutoGluon indeed used multiple models in its final ensemble.
- **Predict and Save**: Predictions are made on `full_test`, and a new submission file `submission_foundation_ensemble.csv` is created and saved. Details of these predictions are also printed.

In [ ]:
sub = pd.read_csv("/content/drive/MyDrive/DSN_collab_bootcamp/submissions/submission_foundation_ensemble.csv")

print("Shape:", sub.shape)                    # expect (1705, 2)
print("Columns:", sub.columns.tolist())       # expect ['id', 'total_sales']
print("Any NaN:", sub.isna().any().any())     # expect False
print("Min pred:", sub.total_sales.min())     # expect > 0
print("Max pred:", sub.total_sales.max())     # expect reasonable (<15000)
print(sub.head())


Shape: (1705, 2)
Columns: ['id', 'total_sales']
Any NaN: False
Min pred: 158.03759765625
Max pred: 6330.27880859375
          id  total_sales
0  row_00009  2843.196777
1  row_00015  4399.883789
2  row_00019  2902.023438
3  row_00020  3280.971191
4  row_00023  2506.419678


#### Verify Final Submission File

This cell loads the `submission_foundation_ensemble.csv` file and performs several checks:
- **Shape and Columns**: Verifies the DataFrame's dimensions and column names.
- **NaN Check**: Ensures there are no missing (NaN) values in the predictions.
- **Prediction Range**: Prints the minimum and maximum prediction values to quickly assess if the output scale is reasonable.
- **Head Display**: Shows the first few rows of the final submission, similar to previous steps.